In [1]:
!pip install sentence-transformers IsoScore

## Set the HF Token

In [4]:
import os
from kaggle_secrets import UserSecretsClient

# Retrieve the secret from Kaggle
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Set it as an environment variable
os.environ["HF_TOKEN"] = HF_TOKEN

## Run the gemma embedding model

In [5]:
"""
IsoScore for google/embeddinggemma-300m.

Computes:
  1. Sentence-level IsoScore on the final pooled embeddings (what model.encode returns)
  2. Token-level IsoScore after every transformer layer (the setup in your Metrics section)

IsoScore uses the official implementation by Rudman et al. (2022):
  pip install IsoScore sentence-transformers
"""

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from IsoScore import IsoScore

def isoscore(points: np.ndarray) -> float:
    """points: (num_samples, dim). Returns IsoScore in [0, 1] via the official library.
    Cast to float64 so results don't depend on the input dtype."""
    X = np.asarray(points, dtype=np.float64)
    if X.shape[0] < 2:
        raise ValueError("Need at least 2 points.")
    return float(IsoScore.IsoScore(X))


def isotropic_ceiling(n: int, d: int) -> float:
    """Approximate IsoScore a perfectly isotropic cloud would get with only n points."""
    return n / (n + d)


model = SentenceTransformer("google/embeddinggemma-300m")

sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day",
]

# ---------------------------------------------------------------
# 1. Sentence-level IsoScore (pooled, projected, normalized output)
# ---------------------------------------------------------------
embeddings = model.encode(sentences)  # shape (4, 768)
n, d = embeddings.shape
print(f"Sentence embeddings: n={n}, d={d}")
print(f"  IsoScore            = {isoscore(embeddings):.6f}")
print(f"  Isotropic ceiling   ~ {isotropic_ceiling(n, d):.6f}  (max achievable with n={n})")

# ---------------------------------------------------------------
# 2. Token-level IsoScore after each transformer layer
#    Tokens from ALL sentences are pooled into one point cloud per layer.
# ---------------------------------------------------------------
backbone = model[0].auto_model          # underlying Gemma3 encoder
tokenizer = model.tokenizer
device = model.device

features = model.preprocess(sentences)
print(features.values())
features = {k: v.to(device) for k, v in features.items() if isinstance(v, torch.Tensor)}

with torch.no_grad():
    out = backbone(
        input_ids=features["input_ids"],
        attention_mask=features["attention_mask"],
        output_hidden_states=True,
    )

# Keep real tokens only: drop padding and special tokens (BOS/EOS carry outlier norms)
input_ids = features["input_ids"]
special_ids = torch.tensor(tokenizer.all_special_ids, device=device)
keep = features["attention_mask"].bool() & ~torch.isin(input_ids, special_ids)

print(f"\nToken-level (pooled real tokens: {int(keep.sum())})")
print(f"{'layer':>5} | {'IsoScore':>10} | {'ceiling':>8}")
# hidden_states[0] = embedding layer output, [i] = after transformer layer i
for layer_idx, hs in enumerate(out.hidden_states):
    pts = hs[keep].float().cpu().numpy()   # (num_tokens, hidden_dim)
    print(f"{layer_idx:>5} | {isoscore(pts):>10.6f} | "
          f"{isotropic_ceiling(*pts.shape):>8.4f}")

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Sentence embeddings: n=4, d=768
  IsoScore            = 0.001005
  Isotropic ceiling   ~ 0.005181  (max achievable with n=4)
ValuesView({'input_ids': tensor([[    2,  6372,   563,   496,  5293,  1589,     1,     0],
        [    2,  6372,   563,   496,  5293,  4799,     1,     0],
        [    2,  6372,   563,   496,  1401,  5293,  1589,     1],
        [    2, 17076,   563,   496, 17635,  1719,     1,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0]]), 'modality': 'text'})

Token-level (pooled real tokens: 21)
layer |   IsoScore |  ceiling
    0 |   0.007755 |   0.0266
    1 |   0.006326 |   0.0266
    2 |   0.003578 |   0.0266
    3 |   0.004444 |   0.0266
    4 |   0.003665 |   0.0266
    5 |   0.004081 |   0.0266
    6 |   0.003740 |   0.0266
    7 |   0.004567 |   0.0266
    8 |   0.004173 |   0.0266
    9 |   0.005381 |   0.0266
   10 |   0.006974 |   0.0266
   11 |  